# Encode ViDoRe pages and queries with ColQwen2.5

Encodes every page and every English query of the development subset (`vidore_v3_hr`) and the
held-out subset (`vidore_v3_finance_en`) with `vidore/colqwen2.5-v0.2` in fp16, using this
project's own encoder and vector store (`mmrag.visual`).

**Kaggle settings:** Accelerator **GPU T4 x2**, Internet **On**. Then *Save Version → Save & Run All*.

**Output:** `/kaggle/working/vectors.zip`. Unzip it into the project's `data/vectors/` to get
`data/vectors/<subset>/{pages.npz, queries.npz, meta.json}`.

A re-run in the same session skips pages and queries that are already encoded.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/yuecao365/multimodal-doc-rag"
COLPALI_ENGINE = "colpali-engine==0.3.18"  # same version as the local visual environment

MODEL = os.environ.get("MMRAG_MODEL", "vidore/colqwen2.5-v0.2")
SUBSETS = os.environ.get("MMRAG_SUBSETS", "vidore/vidore_v3_hr,vidore/vidore_v3_finance_en").split(
    ","
)
LIMIT = int(os.environ.get("MMRAG_LIMIT", "0")) or None  # pages per subset; None = all
BATCH_SIZE = 4

# Off Kaggle (a local trial run) the notebook uses an existing checkout and environment.
ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working" if ON_KAGGLE else os.environ.get("MMRAG_WORK", "kaggle_working"))
REPO = Path("/kaggle/temp/multimodal-doc-rag" if ON_KAGGLE else os.environ.get("MMRAG_REPO", ".."))
VECTORS = WORK / "vectors"
print(f"model={MODEL}  subsets={SUBSETS}  limit={LIMIT}  out={VECTORS}")

In [ ]:
import subprocess
import sys

if ON_KAGGLE:
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
    # Only what encoding needs; pyarrow, huggingface-hub, pillow, numpy and scipy are preinstalled.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", COLPALI_ENGINE], check=True)
    # Kaggle ships torchao 0.10; peft refuses it while loading ColQwen's LoRA adapter
    # ("only versions above 0.16.0 are supported"). Encoding does not use torchao.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)

sys.path.insert(0, str(REPO.resolve() / "src"))
commit = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print("repo commit:", commit)

In [ ]:
import time

import torch

assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2 in the notebook settings"
print(torch.__version__, torch.cuda.get_device_name(0))
# Fails here, not an hour in, if this torch build cannot run on the GPU (old P100 images).
assert torch.isfinite(torch.ones(8, device="cuda", dtype=torch.float16).sum())

from mmrag.visual.encoder import ColpaliEncoder

started = time.perf_counter()
encoder = ColpaliEncoder(MODEL, device="cuda", batch_size=BATCH_SIZE, dtype="float16")
print(f"loaded {MODEL} in {encoder.dtype} in {time.perf_counter() - started:.0f} s")

In [ ]:
from mmrag.visual.encode_cli import encode_dataset

# The T4 has no bfloat16, and fp16 can overflow on some pages. A batch with NaN or inf is
# re-encoded page by page; a page that still overflows is encoded by the same model in
# float32 on CPU (about 15 GB of RAM, loaded only if needed). meta.json lists those pages.
def cpu_fp32_encoder():
    return ColpaliEncoder(MODEL, device="cpu", batch_size=1, dtype="float32")


for dataset in SUBSETS:
    subset = dataset.replace("\\", "/").rstrip("/").split("/")[-1]
    print(f"\n=== {dataset} -> {VECTORS / subset} ===", flush=True)
    encode_dataset(
        encoder, dataset, VECTORS / subset, MODEL,
        limit=LIMIT, batch_size=BATCH_SIZE, fallback=cpu_fp32_encoder,
    )

In [ ]:
import shutil

import numpy as np

from mmrag.data.vidore import load_vidore_subset
from mmrag.visual.store import MultiVectorStore, text_key

# Read every file back the way the retriever will, before packing.
for dataset in SUBSETS:
    subset = dataset.replace("\\", "/").rstrip("/").split("/")[-1]
    data = load_vidore_subset(dataset, n_queries=None)
    pages = MultiVectorStore.load(VECTORS / subset / "pages.npz")
    queries = MultiVectorStore.load(VECTORS / subset / "queries.npz")
    assert pages.ids == data.corpus_ids[:LIMIT], f"{subset}: pages missing or out of order"
    assert all(text_key(q) in queries for q in data.queries), f"{subset}: queries missing"
    assert np.isfinite(pages.vectors).all() and np.isfinite(queries.vectors).all()
    assert pages.meta["model"] == MODEL and pages.quantization == "float16"
    print(subset, "pages", pages.stats(), "queries", len(queries))

archive = shutil.make_archive(str(WORK / "vectors"), "zip", root_dir=VECTORS)
print(f"wrote {archive}  ({Path(archive).stat().st_size / 1e6:.0f} MB)")
if ON_KAGGLE:
    shutil.rmtree(VECTORS)  # keep only the zip in the notebook output